# Bronze Layer – ERA5-Land Ingestion

#### Objective
Ingest raw ERA5-Land monthly data into the Bronze layer
while preserving the original source data.

#### Input
- ERA5-Land monthly data (2000–2025)

#### Output
- Bronze NetCDF dataset
- Ingestion validation summary

Import libraries

In [1]:
from pathlib import Path
import shutil
import json

import xarray as xr
import pandas as pd

Project configuration

In [10]:
PROJECT_DIR = Path.cwd().parent

RAW_FILE = PROJECT_DIR / "data/raw/era5_land_monthly_2000_2025.nc"

BRONZE_DIR = PROJECT_DIR / "data/bronze/era5_land"

BRONZE_FILE = BRONZE_DIR / "era5_land_monthly_2000_2025.nc"

METADATA_FILE = BRONZE_DIR / "metadata.json"

In [11]:
print("Project directory:", PROJECT_DIR)
print("Raw file exists:", RAW_FILE.exists())
print("Bronze directory:", BRONZE_DIR)

Project directory: /home/trang/Drought_prediction
Raw file exists: True
Bronze directory: /home/trang/Drought_prediction/data/bronze/era5_land


Source Data Inspection

In [ ]:
#Inspect the original ERA5-Land NetCDF dataset before ingestion.
with xr.open_dataset(RAW_FILE) as ds:
    print("Dataset information:")
    print(ds)

    print("\nData variables:")
    for var in ds.data_vars:
        print(
            f"{var}: "
            f"long_name={ds[var].attrs.get('long_name')}, "
            f"units={ds[var].attrs.get('units')}"
        )

    print("\nDimensions:")
    print(ds.sizes)

    print("\nCoordinates:")
    print(list(ds.coords))

Dataset information:
<xarray.Dataset> Size: 12MB
Dimensions:     (valid_time: 312, latitude: 32, longitude: 38)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2kB 2000-01-01 ... 2025-12-01
    expver      (valid_time) <U4 5kB ...
  * latitude    (latitude) float64 256B 11.1 11.0 10.9 10.8 ... 8.3 8.2 8.1 8.0
  * longitude   (longitude) float64 304B 103.3 103.4 103.5 ... 106.8 106.9 107.0
    number      int64 8B ...
Data variables:
    d2m         (valid_time, latitude, longitude) float32 2MB ...
    t2m         (valid_time, latitude, longitude) float32 2MB ...
    swvl1       (valid_time, latitude, longitude) float32 2MB ...
    swvl2       (valid_time, latitude, longitude) float32 2MB ...
    swvl3       (valid_time, latitude, longitude) float32 2MB ...
    swvl4       (valid_time, latitude, longitude) float32 2MB ...
    pev         (valid_time, latitude, longitude) float32 2MB ...
    tp          (valid_time, latitude, longitude) float32 2MB ...
Attributes:
    GRIB_centr

Data Ingestion

In [6]:
# Copy raw ERA5-Land data into Bronze layer

if not RAW_FILE.exists():
    raise FileNotFoundError(f"Source file not found: {RAW_FILE}")

if not BRONZE_FILE.exists():
    shutil.copy2(RAW_FILE, BRONZE_FILE)
    print("Data ingestion completed.")
else:
    print("Bronze file already exists. Skipping ingestion.")

print("Source:", RAW_FILE)
print("Destination:", BRONZE_FILE)
print("Bronze file size:", round(BRONZE_FILE.stat().st_size / (1024**2), 2), "MB")

Data ingestion completed.
Source: /home/trang/Drought_prediction/data/raw/era5_land_monthly_2000_2025.nc
Destination: /home/trang/Drought_prediction/data/bronze/era5_land/era5_land_monthly_2000_2025.nc
Bronze file size: 3.2 MB


Ingestion Verification

In [ ]:
# Verify that the ingested Bronze dataset exists
# and can be opened successfully.

with xr.open_dataset(BRONZE_FILE) as ds:
    print("Bronze dataset loaded successfully.")
    print("Dimensions:", dict(ds.sizes))
    print("Variables:", list(ds.data_vars))

Bronze dataset loaded successfully.
Dimensions: {'valid_time': 312, 'latitude': 32, 'longitude': 38}
Variables: ['d2m', 't2m', 'swvl1', 'swvl2', 'swvl3', 'swvl4', 'pev', 'tp']


Ingestion metadata

In [8]:
metadata = {
    "dataset": "ERA5-Land Monthly",
    "source_file": str(RAW_FILE.relative_to(PROJECT_DIR)),
    "bronze_file": str(BRONZE_FILE.relative_to(PROJECT_DIR)),
    "format": "NetCDF",
    "file_size_mb": round(BRONZE_FILE.stat().st_size / (1024**2), 2),
    "time_range": "2000-01 to 2025-12",
    "frequency": "Monthly",
    "ingestion_status": "success"
}

with open(METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print(json.dumps(metadata, indent=4, ensure_ascii=False))

{
    "dataset": "ERA5-Land Monthly",
    "source_file": "data/raw/era5_land_monthly_2000_2025.nc",
    "bronze_file": "data/bronze/era5_land/era5_land_monthly_2000_2025.nc",
    "format": "NetCDF",
    "file_size_mb": 3.2,
    "time_range": "2000-01 to 2025-12",
    "frequency": "Monthly",
    "ingestion_status": "success"
}


In [12]:
bronze_ds = xr.open_dataset(BRONZE_FILE)

print("Bronze dimensions:", bronze_ds.sizes)
print("Recovery dimensions:", recovery_full.sizes)

print(
    "Same latitude:",
    np.array_equal(bronze_ds.latitude.values, recovery_full.latitude.values)
)

print(
    "Same longitude:",
    np.array_equal(bronze_ds.longitude.values, recovery_full.longitude.values)
)

print(
    "Same variables:",
    set(["tp", "pev"]).issubset(bronze_ds.data_vars)
)

Bronze dimensions: Frozen({'valid_time': 312, 'latitude': 32, 'longitude': 38})


NameError: name 'recovery_full' is not defined